In [2]:
from importlib.metadata import version
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
enc_text_full = tokenizer.encode (raw_text)
enc_text = enc_text_full[50:]

In [5]:
context_size = 4

In [6]:
x = enc_text[:context_size]
y = enc_text[1:context_size + 1]

In [7]:
print(f"x: {x}")
print(f"y:     {y}")

x: [290, 4920, 2241, 287]
y:     [4920, 2241, 287, 257]


In [8]:
#prediction simulation
for i in range(1, context_size + 1):
    context = enc_text[:i]
    next_token = enc_text[i]
    print(f"{tokenizer.decode(context)} -- > {tokenizer.decode([next_token])}")

 and -- >  established
 and established -- >  himself
 and established himself -- >  in
 and established himself in -- >  a


In [12]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids), stride):
            input_chunk = token_ids[i:i+stride]
            target_chunk = token_ids[i + 1:i + stride + 1]
            self.input_ids.append(input_chunk)
            self.target_ids.append(target_chunk)
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_szie = 4, max_length=56, stride=128, shuffle=True, drop_last=True, num_workers=0):
    